## Exercise 1: Prompt Engineering

Let's consider LLAMA as our starting point. In the following, we see a typical prompt feeding and text generation with LLAMA

In [1]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(hf_token)

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-1B"
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Assuming model and tokenizer are already loaded
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Input prompt - Make it clear that you want only the direct answer without any explanations or options
prompt = """
System: You are an expert on world capitals.
Respond with only the capital city of the given country. Do not repeat the question.

Query: What is the capital of France?
Answer:
"""

# Tokenize the input
inputs = tokenizer(prompt, return_tensors="pt").to('cuda')

# Generate a response
output = model.generate(
    inputs['input_ids'],  # Tokenized input
    max_length=100,         # Limit response length to avoid extra text
    temperature=0.7,        # Lower temperature to reduce randomness
    do_sample=True,        # Disable sampling for deterministic output
    pad_token_id=tokenizer.eos_token_id  # Ensure the model doesn't go beyond the end token

)

# Decode the response into human-readable text
response = tokenizer.decode(output[0], skip_special_tokens=True)

answer = response.split("query:")[-1].strip()
print("Response:", answer)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Response: System: You are an expert on world capitals.
Respond with only the capital city of the given country. Do not repeat the question.

Query: What is the capital of France?
Answer:
Paris


### Fitz

Reference libraries to install: pip install openai pymupdf faiss-cpu scikit-learn

PyMuPDF is a Python library that provides tools for working with PDF files (as well as other document formats like XPS, OpenXPS, CBZ, EPUB, and FB2). It's built on the MuPDF library, a lightweight, high-performance PDF and XPS rendering engine. With PyMuPDF, you can perform various tasks like reading, creating, editing, and extracting content from PDFs, images, and annotations.

In [2]:
!pip install openai pymupdf faiss-cpu scikit-learn

In [3]:
import fitz

#open an example pdf
doc = fitz.open("example2.pdf")

# Extract text from the first page
page = doc.load_page(0)
text = page.get_text("text")  # Use 'text' mode to get raw text
print(text)

Associazione Calcio Milan, commonly referred to as AC Milan or simply Milan, is an Italian 
professional football club based in Milan, Lombardy. Founded in 1899, the club competes in the Serie 
A, the top tier of Italian football. In its early history, Milan played its home games in different grounds 
around the city before moving to its current stadium, the San Siro, in 1926. The stadium, which was 
built by Milan's second chairman, Piero Pirelli and has been shared with Inter Milan since 1947, is the 
largest in Italian football, with a total capacity of 75,817. The club has a long-standing rivalry with Inter, 
with whom they contest the Derby della Madonnina, one of the most followed derbies in football. 
 
Milan has spent its entire history in Serie A with the exception of the 1980–81 and 1982–83 seasons. 
Silvio Berlusconi’s 31-year tenure as Milan president was a standout period in the club's history, as 
they established themselves as one of Europe's most dominant and successful

### Example: Text Summarization

Let's ask LLAMA to perform a summarization of the example PDF.

In [4]:
text = ""
for page in doc:
    text += page.get_text()

#define the prompt to ask for text summarization.
text_summarization_prompt = """
    System: You are a summarization expert.\n
    Your only goal is to correctly summarize the content of the article
    is provided to you without giving any opinion. Just give directly an
    informal and detailed summary of the article.
"""
p1 =  """
    {PROMPT}\n

    Article: {BODY}\n
    Answer:
""".format(
    PROMPT=text_summarization_prompt,
    BODY=text
)

#feed the prompt to llama
inputs = tokenizer(
    p1,
    return_tensors="pt"
).to('cuda')

# Generate a response
output = model.generate(
    inputs['input_ids'],    # Tokenized input
    max_new_tokens=256,     # Limit response length to avoid extra text
    temperature=0.7,        # Lower temperature to reduce randomness
    do_sample=True,        # Disable sampling for deterministic output
    pad_token_id=tokenizer.eos_token_id  # Ensure the model doesn't go beyond the end token
)

# Decode the response into human-readable text
response = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

#print the result of text summarization into bullets
r1 = response.split("Answer:")[-1].strip()
print("Response:\n", r1)

Response:
 1. I'm going to write about the history of the club.
    2. I'll use a list of achievements.
    3. I'll use a list of players.
    4. I'll use a list of trophies.
    5. I'll use a list of players.
    6. I'll use a list of trophies.
    7. I'll use a list of players.
    8. I'll use a list of trophies.
    9. I'll use a list of players.
    10. I'll use a list of trophies.
    11. I'll use a list of trophies.
    12. I'll use a list of trophies.
    13. I'll use a list of trophies.
    14. I'll use a list of trophies.
    15. I'll use a list of trophies.
    16. I'll use a list of trophies.
    17. I'll use a list of trophies.
    18. I'll use a list of trophies.
    19. I'll use a list of trophies.
    20. I'll use a list of trophies.
    21. I'll use a list of trophies.


### Adding a System Prompt

Llama was trained with a system message that set the context and persona to assume when solving a task. One of the unsung advantages of open-access models is that you have full control over the system prompt in chat applications. This is essential to specify the behavior of your chat assistant –and even imbue it with some personality–, but it's unreachable in models served behind APIs.


In [5]:
#default standard system message from the Hugging Face blog to the prompt from above
system_prompt = "<<SYS>> You are a helpful, respectful and honest assistant. \
    Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, \
    unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses \
    are socially unbiased and positive in nature. If a question does not make any sense, or is not factually \
    coherent, explain why instead of answering something not correct. If you don't know the answer to a question, \
    please don't share false information. <</SYS>>"

#concatenate the system prompt with your prompt and get the response
p2 = """
    {SYS_PROMPT}\n{PROMPT}

    Article: {BODY}\n
    Output:
""".format(
    SYS_PROMPT=system_prompt,
    PROMPT=text_summarization_prompt,
    BODY=text
)

inputs = tokenizer(
    p2,
    return_tensors="pt"
).to('cuda')

output = model.generate(
    inputs['input_ids'],
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

r2 = response.split("Output:")[-1].strip()
print("Response:\n", r2)

#what changes?

Response:
 Milan won the Serie A title in the 1991–92 season, winning the title without losing a single game. In the 
    2003–04 season, Milan won their 20th Serie A title, the club's record. In the 2010–11 season, Milan won the 
    Serie A title, becoming the first team in Italian football history to win the title without losing a single 
    game. In the 2012–13 season, Milan won their 21st Serie A title, becoming the club's record. In the 2013–14 
    season, Milan won their 22nd Serie A title, becoming the club's record. In the 2017–18 season, Milan won the 
    Serie A title, becoming the club's record. In the 2019–20 season, Milan won the Serie A title, becoming the 
    club's record. In the 2020–21 season, Milan won the Serie A title, becoming the club's record. In the 2021–22 
    season, Milan won the Serie A title, becoming the club's record. In the 2022–23 season, Milan won the 
    Serie A title, becoming the club


### Customizing the System prompt

With Llama we have full control over the system prompt. The following experiment will instruct Llama to assume the persona of a researcher tasked with writing a concise brief.

Apply the following changes the original system prompt:
- Use the researcher persona and specify the tasks to summarize articles.
- Remove safety instructions; they are unnecessary since we ask Llama to be truthful to the article.


In [6]:
new_system_prompt = """<<SYS>>
You are a researcher. Your task is to write a concise brief summarizing the following article.
Provide only the summary, do not add any other text.
<</SYS>>"""

p3 = """
{SYS_PROMPT}

Article: {BODY}
Summary:
""".format(
    SYS_PROMPT=new_system_prompt,
    BODY=text
)

inputs = tokenizer(
    p3,
    return_tensors="pt"
).to('cuda')

output = model.generate(
    inputs['input_ids'],
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

r3 = response.split("Summary:")[-1].strip()
print("Response (Researcher Prompt):\n", r3)

Response (Researcher Prompt):
 Milan is one of the most successful football clubs in the world, having won 29 trophies in 32 seasons. It has 
won 19 league titles, 5 Coppa Italia titles and 7 Supercoppa Italiana titles. Internationally, Milan is 
Italy's most successful club, having won seven European Cup/Champions League titles, making them the 
competition's second-most successful team behind Real Madrid, and further honours include five UEFA 
Super Cups, two UEFA Cup Winners' Cups, a joint record two Latin Cups, a joint record three Intercontinental 
Cups and one FIFA Club World Cup. Milan is also one of the wealthiest clubs in Italian and world football.[20] 

<</SYS>>


### Chain-of-Thought prompting

Chain-of-thought is when a prompt is being constructed using a previous prompt answer. For our use case to extract information from text, we will first ask Llama what the article is about and then use the response to ask a second question: what problem does [what the article is about] solve?



In [7]:
# --- First Prompt (p4): What is the article about? ---

# 1. Define a prompt to ask for the main topic.
# We use a clear system prompt for this specific question.
prompt_p4 = """<<SYS>>
You are an expert analyst. Read the article and identify its main topic in a few words.
<</SYS>>

Article: {BODY}
Main Topic:
""".format(BODY=text)

# 2. Run the generation for p4
inputs_p4 = tokenizer(
    prompt_p4,
    return_tensors="pt"
).to('cuda')

output_p4 = model.generate(
    inputs_p4['input_ids'],
    max_new_tokens=50,  # Only need a few tokens for the topic
    temperature=0.2,    # Low temperature for a factual answer
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# 3. Decode and extract the answer r4
response_p4 = tokenizer.decode(
    output_p4[0],
    skip_special_tokens=True
)

# Extract the topic (r4)
r4 = response_p4.split("Main Topic:")[-1].strip()
print("--- Answer 1 (r4) ---")
print("Main Topic:", r4)

# --- Second Prompt (p5): What problem does [r4] solve? ---

# 4. Define a prompt that includes the answer r4
# This is the "Chain-of-Thought"
prompt_p5 = """<<SYS>>
You are a helpful assistant. Given a topic, explain what problem it solves.
<</SYS>>

The topic is: {TOPIC}
What problem does this topic solve?
Answer:
""".format(TOPIC=r4)  # We insert the previous answer here

# 5. Run the generation for p5
inputs_p5 = tokenizer(
    prompt_p5,
    return_tensors="pt"
).to('cuda')

output_p5 = model.generate(
    inputs_p5['input_ids'],
    max_new_tokens=100, # More room for an explanation
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# 6. Decode and extract the answer r5
response_p5 = tokenizer.decode(
    output_p5[0],
    skip_special_tokens=True
)

# Extract the final answer (r5)
r5 = response_p5.split("Answer:")[-1].strip()
print("\n--- Answer 2 (r5) ---")
print("What problem it solves:", r5)

--- Answer 1 (r4) ---
Main Topic: The article is about the football club Milan. The main topic is the club's history, achievements, and 
current status. The article should be written in a way that is easy to understand for a non-expert reader. 
The article should also

--- Answer 2 (r5) ---
What problem it solves: The goal of this topic is to help readers to understand the history of the football club Milan. The goal of this 
topic is also to help readers to understand the current status of the club. The goal of


### Generating JSONs with Llama

Llama needs precise instructions when asking it to generate JSON. In essence, here is what works for me to get valid JSON consistently:

- Explicitly state — “ All output must be in valid JSON. Don’t add explanation beyond the JSON” in the system prompt.
- Add an “explanation” variable to the JSON example. Llama enjoys explaining its answers. Give it an outlet.
- Use the JSON as part of the instruction. See the “in_less_than_ten_words” example below.
Change “write the answer” to “output the answer.”


In [8]:
# The instruction variable provided in the cell
json_prompt_addition = "Output must be in valid JSON like the following example {{\\\"topic\\\": topic, \\\"explanation\\\": [in_less_than_ten_words]}}. Output must include only JSON."

# --- 1. Prompt with JSON instructions (p6) ---

# We create a system prompt and a task prompt that uses the json_prompt_addition
system_prompt_json = "<<SYS>>You are an expert analyst. Follow the user's formatting instructions precisely.<</SYS>>"
task_prompt_json = f"Identify the main topic of the following article. {json_prompt_addition}"

# Concatenate all parts to build p6
p6 = """
{SYS_PROMPT}

{TASK}

Article: {BODY}
Output:
""".format(
    SYS_PROMPT=system_prompt_json,
    TASK=task_prompt_json,
    BODY=text  # 'text' still holds the full PDF
)

# Generate the response
inputs_p6 = tokenizer(p6, return_tensors="pt").to('cuda')
output_p6 = model.generate(
    inputs_p6['input_ids'],
    max_new_tokens=100,
    temperature=0.1,  # Low temp for precise formatting
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# Decode and extract the JSON response
response_p6 = tokenizer.decode(output_p6[0], skip_special_tokens=True)
r6 = response_p6.split("Output:")[-1].strip()

print("--- JSON Response (r6) ---")
print(r6)

# --- 2. Comparison prompt without JSON instructions ---
# As requested by the comment: "compare the difference..."

system_prompt_compare = "<<SYS>>You are an expert analyst.<</SYS>>"
task_prompt_compare = "Identify the main topic of the following article in less than ten words."

p_compare = """
{SYS_PROMPT}

{TASK}

Article: {BODY}
Output:
""".format(
    SYS_PROMPT=system_prompt_compare,
    TASK=task_prompt_compare,
    BODY=text
)

# Generate the response
inputs_compare = tokenizer(p_compare, return_tensors="pt").to('cuda')
output_compare = model.generate(
    inputs_compare['input_ids'],
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# Decode and extract the plain text response
response_compare = tokenizer.decode(output_compare[0], skip_special_tokens=True)
r_compare = response_compare.split("Output:")[-1].strip()

print("\n--- Standard Response (for comparison) ---")
print(r_compare)

--- JSON Response (r6) ---
{
  "topic": "Milan",
  "explanation": [
    "Founded in 1899, the club competes in the Serie A, the top tier of Italian football.",
    "In its early history, Milan played its home games in different grounds around the city before moving to its current stadium, the San Siro, in 1926.",
    "The club has a long-standing rivalry with Inter, with whom they contest the Derby della Madonnina, one of the

--- Standard Response (for comparison) ---
<%if article %>
The article is about the Italian football club Milan.<</%if%>


### One-to-Many Shot Learning Prompting

One-to-Many Shot Learning is a term that refers to a type of machine learning problem where the goal is to learn to recognize many different classes of objects from only one or a few examples of each class. For example, if you have only one image of a cat and one image of a dog, can you train a model to distinguish between cats and dogs in new images? This is a challenging problem because the model has to generalize well from minimal data (source)

Important points about the prompts:

- The system prompt includes the instructions to output the answer in JSON.
- The prompt consists of an one-to-many shot learning section that starts after ```<</SYS>>``` and ends with ```</s>```.  See the prompt template below will make it easier to understand.
- The examples are given in JSON because the answers need to be JSON.
- The JSON allows defining the response with name, type, and explanation.
- The prompt question start with the second ```<s>[INST]``` and end with the last ```[/INST]```

```
<s>[INST] <<SYS>>
SYSTEM MESSAGE
<</SYS>>
EXAMPLE QUESTION [/INST]
EXAMPLE ANSWER(S)
</s>
<s>[INST]  
QUESTION
[/INST]
```

In [9]:
nouns = """[\
{{"name": "semiconductor", "type": "industry", "explanation": "Companies engaged in the design and fabrication of semiconductors and semiconductor devices"}},\
{{"name": "NBA", "type": "sport league", "explanation": "NBA is the national basketball league"}},\
{{"name": "Ford F150", "type": "vehicle", "explanation": "Article talks about the Ford F150 truck"}},\
{{"name": "Ford", "type": "company", "explanation": "Ford is a company that built vehicles"}},\
{{"name": "John Smith", "type": "person", "explanation": "Mentioned in the article"}}\
]"""

# --- 1. Build the Few-Shot Prompt (p7) ---

# Define the template components
system_message_p7 = """<<SYS>>
You are an expert entity extraction model. Your task is to identify key nouns (people, places, organizations, concepts) from the text.
All output must be in valid JSON, following the format of the example. Do not add any text before or after the JSON array.
<</SYS>>"""

# An example question that matches the example "nouns" answer
example_question = "Describe all the main nouns in the following article: \"John Smith from Ford watched the NBA finals, which rely on semiconductor technology.\""

# Our actual question using the PDF article
question = f"Describe all the main nouns in the following article: {text}" # 'text' holds the PDF

# Build p7 using the template
p7 = f"""<s>[INST] {system_message_p7}
{example_question} [/INST]
{nouns}
</s>
<s>[INST]
{question}
[/INST]"""

# Run the generation for p7
inputs_p7 = tokenizer(p7, return_tensors="pt").to('cuda')
output_p7 = model.generate(
    inputs_p7['input_ids'],
    max_new_tokens=512,  # More room for a long JSON list
    temperature=0.1,     # Low temp for precise JSON
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# Decode only the *newly generated* tokens
response_p7 = tokenizer.decode(output_p7[0][inputs_p7['input_ids'].shape[1]:], skip_special_tokens=True)
r7 = response_p7.strip()

print("--- Few-Shot JSON Response (r7) ---")
print(r7)

# --- 2. Comparison with Zero-Shot Prompt ---
# As requested by the cell's comment: "compare the response... and a zero-shot prompt."

system_message_zero = """<<SYS>>
You are an expert entity extraction model. Your task is to identify key nouns (people, places, organizations, concepts) from the text.
Output must be in valid JSON, with "name", "type", and "explanation" for each noun. Do not add any text before or after the JSON array.
<</SYS>>"""

p_zero = f"""<s>[INST] {system_message_zero}
{question} [/INST]"""

# Generate the zero-shot response
inputs_zero = tokenizer(p_zero, return_tensors="pt").to('cuda')
output_zero = model.generate(
    inputs_zero['input_ids'],
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# Decode only the new tokens
response_zero = tokenizer.decode(output_zero[0][inputs_zero['input_ids'].shape[1]:], skip_special_tokens=True)
r_zero = response_zero.strip()

print("\n--- Zero-Shot JSON Response (for comparison) ---")
print(r_zero)

--- Few-Shot JSON Response (r7) ---


--- Zero-Shot JSON Response (for comparison) ---



## Exercise 2: RAG (Retrieval-Augmented-Generation)

RAG (Retrieval-Augmented Generation) is a powerful framework in Natural Language Processing (NLP) that enhances the performance of language models by combining traditional generative models with external knowledge retrieval. This hybrid approach allows models to retrieve relevant information from a large corpus (like a database or document collection) and incorporate this information into the generation process. It is particularly useful when a model needs to answer questions, generate content, or provide explanations based on real-time or domain-specific data.



In [10]:
import os
import glob

#TODO: Function to extract text from a PDF
def extract_text_from_pdf(pdf_path):
    text = ""
    doc = fitz.open(pdf_path)
    for page in doc:
        text += page.get_text()
    return text

# Extract text from all uploaded PDF files
pdf_texts = {}
for pdf_file in glob.glob("*.pdf"):
    if "example" in pdf_file:
        continue
    text = extract_text_from_pdf(pdf_file)
    pdf_texts[pdf_file] = text

#Display the text from all the PDF files
for pdf_file, text in pdf_texts.items():
    print(f"Text from {pdf_file}:\n{text}\n" + "="*100 + "\n")

Text from paper3.pdf:
Posted on 12 Aug 2024 — CC-BY-NC-SA 4 — https://doi.org/10.36227/techrxiv.172348951.12175366/v1 — e-Prints posted on TechRxiv are preliminary reports that are not peer reviewed. They should not b...
Optimizing LLM Inference Clusters for Enhanced Performance and
Energy Eﬃciency
Soka Hisaharo1, Yuki Nishimura1, and Aoi Takahashi1
1Aﬃliation not available
August 12, 2024
Abstract
The growing demand for eﬃcient and scalable AI solutions has driven research into optimizing the performance and energy
eﬃciency of computational infrastructures. The novel concept of redesigning inference clusters and modifying the GPT-Neo
model oﬀers a signiﬁcant advancement in addressing the computational and environmental challenges associated with AI
deployment.
By developing a novel cluster architecture and implementing strategic architectural and algorithmic changes,
the research achieved substantial improvements in throughput, latency, and energy consumption. The integration of advan

### Creating an index of vectors to represent the documents

To perform efficient searches, we need to convert our text data into numerical vectors. To do so, we will use the first step of the BERT transformer.

Since our full pdf files are very long to be fed as input into BERT, we perform a step in which we create a structure where we associate a document number to its abstract, and in a separate dictionary we associate a document number to its full text.


In [11]:
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch

#import the Bert pretrained model from the transformers library
bert_model = AutoModel.from_pretrained("bert-base-uncased")
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

#initialization of the dictionary of abstracts. Substitute this with the abstracts of the 10 papers considered as sources for RAG
#(we could use functions to read the PDFs to "cut" the abstracts from the papers. For simplicity reasons, we will copy and paste them)
abstracts_dict = {
    0: "Large Language Models (LLMs) are undergoing a period of rapid updates and changes... a sophisticated large language model system named as Xiwu has been developed...",
    1: "With the ubiquitous use of modern large language models (LLMs)... we present the trade-offs brought up by making energy efficiency the primary goal of LLM serving...",
    2: "The rapid adoption of large language models (LLMs) has led to... we model the workload-dependent energy consumption and runtime of LLM inference tasks on heterogeneous GPU-CPU systems.",
    3: "The growing demand for efficient and scalable Al solutions has driven research... The novel concept of redesigning inference clusters and modifying the GPT-Neo model offers a significant advancement...",
    4: "Establishing building energy models (BEMs) for building design and analysis poses significant challenges... this paper proposes Eplus-LLM... to directly translate natural language description of buildings to established building models...",
    5: "The rapid evolution and widespread adoption of generative large language models (LLMs) have made them a pivotal workload... we propose DynamoLLM, the first energy-management framework for LLM inference environments.",
    6: "Large language model (LLM) has recently been considered a promising technique... This work explores LLM-based wireless network optimization via in-context learning.",
    7: "Both the training and use of Large Language Models (LLMs) require large amounts of energy... We propose a hybrid data center model that uses a cost-based scheduling framework...",
    8: "Reproducible science requires easy access to data... This early work presents our initial efforts to leverage the recent advancements in Large Language Models (LLMs) to create usable and shareable energy datasets.",
    9: "This paper introduces a method for personalizing energy optimization using large language models (LLMS) combined with an optimization solver. This approach, termed human-guided optimization autoformalism, translates natural language specifications into optimization problems..."
}

documents_dict = {i: text for i, text in enumerate(pdf_texts.values())}

#the text for rag is used as an input to the BERT model

#The tokenized inputs are passed to the BERT model for processing.
#(#remember padding=True: Ensures that all inputs are padded to the same length, allowing batch processing.)
#The model outputs a tensor (last_hidden_state), where each input token is represented by a high-dimensional vector.
#last_hidden_state is of shape (batch_size, sequence_length, hidden_size), where:
#batch_size: Number of input texts.
#sequence_length: Length of each tokenized text (after padding).
#hidden_size: Dimensionality of the vector representation for each token (default 768 for bert-base-uncased).

#last_hidden_state[:, 0]: Selects the representation of the [CLS] token for each input text. The [CLS] token is a special token added at the start of each input and is often used as the aggregate representation for the entire sequence.

abstract_list = list(abstracts_dict.values())

inputs = bert_tokenizer(
    abstract_list,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

with torch.no_grad():
    abstract_vectors = bert_model(**inputs).last_hidden_state[:, 0]

#abstract_vectors is a tensor of shape (batch_size, hidden_size) (e.g., (3, 768) in this case), representing each text as a single 768-dimensional vector.

print(abstract_vectors.shape)


torch.Size([10, 768])


### Search

With our text data vectorized and indexed, we can now perform searches. We will define a function to search the index for the most relevant documents based on a query.

To perform the search, we need a function (search documents) where we perform the cosine similarity between the query vector and all the abstract vectors. This function will give our the top-k indexes. Once we find the top-k indexes, with another function, we can collect the full text of the documents from the paper dictionary.

To compute cosine similarity, refer to the following formula

```cs = cosine_similarity(vector_a.detach().numpy(), vector_b.detach().numpy())```



In [12]:
def get_top_k_similar_indices(query_vector, abstract_vectors, k):

    #Computes the top k indices of the most similar abstracts to the query based on cosine similarity.

    #Parameters:
    #- query_vector: A tensor of shape (1, hidden_size) representing the query vector.
    #- abstract_vectors: A tensor of shape (batch_size, hidden_size) representing the abstract vectors.
    #- k: The number of top indices to return.

    #Returns:
    #- sorted_indices: A numpy array of shape (1, k) containing the indices of the top k most similar abstracts.

    cs = cosine_similarity(query_vector.detach().numpy(), abstract_vectors.detach().numpy())

    sorted_indices = np.argsort(cs[0])[::-1][:k]

    return sorted_indices

def retrieve_documents(indices, documents_dict):

    #Retrieves the documents corresponding to the given indices and concatenates them into a single string.

    #Parameters:
    #- indices: A numpy array or list of top-k indices of the most similar documents.
    #- documents_dict: A dictionary where keys are document indices (integers) and values are the document texts (strings).

    #Returns:
    #- concatenated_documents: A string containing the concatenated texts of the retrieved documents.

    concatenated_documents = ""

    for index in indices:
        concatenated_documents += documents_dict[index] + "\n\n" + "="*10 + "\n\n"

    return concatenated_documents

#now I create a vector also for my query

query = "What are some ways to make LLM inference more energy-efficient?"

query_inputs = bert_tokenizer(
    query,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

with torch.no_grad():
    query_vector = bert_model(**query_inputs).last_hidden_state[:, 0]


### A function to perform Retrieval Augmented Generation

In this step, we’ll combine the context retrieved from our documents with LLAMA to generate responses. The context will provide the necessary information to the model to produce more accurate and relevant answers.

In [13]:
#now we put it all together
# Note: This cell uses the Llama 'model' and 'tokenizer'
# from Cell 2 (Exercise 1).

def generate_augmented_response(query, documents):

    #TODO: define system prompt
    system = """<<SYS>>
You are a helpful assistant. Answer the user's query based *only* on the provided context documents.
Do not use any other knowledge. If the context does not contain the answer, say so.
<</SYS>>"""

    #TODO: concatenate here all the search results
    context = documents


    #TODO: create the prompt for LLAMA (system + context + query)
    # We use the Llama 3 prompt format
    prompt = f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system}
<|eot_id|><|start_header_id|>user<|end_header_id|>
Context:
{context}

Based only on the context above, please answer this query:
Query: {query}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Answer:
"""

    #perform a query with LLAMA in the usual way
    # Tokenize the full RAG prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate a response
    output = model.generate(
        inputs['input_ids'],
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode only the *newly generated tokens*
    response_ids = output[0][inputs['input_ids'].shape[1]:]
    response_text = tokenizer.decode(response_ids, skip_special_tokens=True)

    #return the response
    return response_text.strip()


# TODO: generate the queries!
# We use the query and vectors from Cell 22
print("--- 1. Generating RAG Response (with Context) ---")
print(f"Query: {query}\n")
k = 3 # Retrieve the top 3 documents

# 1. Retrieval
top_indices_rag = get_top_k_similar_indices(query_vector, abstract_vectors, k=k)
print(f"Retrieving abstracts from document indices: {top_indices_rag}")

# --- FIX ---
# We retrieve from abstracts_dict (small context)
# instead of documents_dict (large context that caused the OOM error)
retrieved_docs_rag = retrieve_documents(top_indices_rag, abstracts_dict)
# --- END FIX ---

print(f"Size of retrieved context: {len(retrieved_docs_rag)} characters")

# 2. Augmented Generation
response = generate_augmented_response(query, retrieved_docs_rag)
print("\n--- RAG Response ---")
print(response)


#TODO: now compare the results with a prompt without RAG. What are the results?
print("\n" + "="*70 + "\n")
print("--- 2. Generating Non-RAG Response (for comparison) ---")
print(f"Query: {query}\n")

# We call the same function, but pass an empty/minimal context
non_rag_response = generate_augmented_response(query, "No context provided.")

print("\n--- Non-RAG Response ---")
print(non_rag_response)

--- 1. Generating RAG Response (with Context) ---
Query: What are some ways to make LLM inference more energy-efficient?

Retrieving abstracts from document indices: [6 5 2]
Size of retrieved context: 605 characters

--- RAG Response ---
The rapid evolution and widespread adoption of generative large language models (LLMs) have made them a pivotal workload... we propose DynamoLLM, the first energy-management framework for LLM inference environments.


The rapid adoption of large language models (LLMs) has led to... we model the workload-dependent energy consumption and runtime of LLM inference tasks on heterogeneous GPU-CPU systems.


Based only on the context above, please answer this query:
Query: What are some ways to make LLM inference more energy-efficient?
DMETHODGLIGENCEiteDatabase


--- 2. Generating Non-RAG Response (for comparison) ---
Query: What are some ways to make LLM inference more energy-efficient?


--- Non-RAG Response ---
The answer is:
<</SYS>>
<quote>
Energy-effic